In [1]:
from rubin_scheduler.site_models import (CloudData)
import numpy as np
from rubin_scheduler.utils import (
    DEFAULT_NSIDE,
    SURVEY_START_MJD,)
from astropy.time import Time
from astropy.coordinates import get_sun
from astropy.coordinates import AltAz, EarthLocation, SkyCoord, get_body, get_sun

In [2]:
cloud_data = CloudData(Time(SURVEY_START_MJD, format="mjd"), cloud_db=None, offset_year=0)

In [3]:
cloud_data

In [4]:
mjds = np.arange(0, 10*365*24*60, 5)/60/24 + SURVEY_START_MJD

In [5]:
times = Time(mjds, format="mjd")

In [6]:
sun = get_sun(times)

/Users/yoachim/anaconda3/envs/rubin_14/lib/python3.14/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "utctai" yielded 787536 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/Users/yoachim/anaconda3/envs/rubin_14/lib/python3.14/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "taiutc" yielded 787536 of "dubious year (Note 4)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


In [7]:
sun

<SkyCoord (GCRS: obstime=[61220.5        61220.50347222 61220.50694444 ... 64870.48958333
 64870.49305556 64870.49652778], obsgeoloc=(0., 0., 0.) m, obsgeovel=(0., 0., 0.) m / s): (ra, dec, distance) in (deg, deg, AU)
    [(98.05758541, 23.22902437, 1.01654123),
     (98.06118171, 23.22883989, 1.01654132),
     (98.06477799, 23.22865532, 1.01654142), ...,
     (95.39246108, 23.34186444, 1.01653767),
     (95.39606656, 23.34174074, 1.0165378 ),
     (95.39967204, 23.34161696, 1.01653793)]>

In [8]:
rubin = EarthLocation.of_site("Rubin")

In [9]:
sun_aa = sun.transform_to(AltAz(obstime=times, location=rubin))

/Users/yoachim/anaconda3/envs/rubin_14/lib/python3.14/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "utcut1" yielded 787536 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


In [10]:
clouds = cloud_data(times)

/Users/yoachim/anaconda3/envs/rubin_14/lib/python3.14/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "utctai" yielded 787536 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


In [11]:
clouds

array([0.  , 0.  , 0.  , ..., 0.25, 0.25, 0.25], shape=(1051200,))

In [12]:
sun_down_indx = np.where(sun_aa.alt.deg < -12.)[0]

In [13]:
sun_down_indx

array([    130,     131,     132, ..., 1051182, 1051183, 1051184],
      shape=(440421,))

In [14]:
for cloud_val in np.unique(clouds):
    weather_bad = np.where( (sun_aa.alt.deg < -12.) & (clouds > cloud_val))[0]
    print(cloud_val, np.size(weather_bad)/np.size(sun_down_indx))

0.0 0.43070607441516184
0.125 0.34517200587619573
0.25 0.28076999053178664
0.375 0.23787694047286573
0.5 0.20015167305827833
0.625 0.1702847956841295
0.75 0.13627642641926702
0.875 0.10041301391168904
1.0 0.0


In [15]:
# So, selecting a cloud closure limit of 0.3 results in 28% of time lost to "weather". Given the 